In [1]:
import os
os.chdir('/home/smallyan/eval_agent')

# Load environment variables from bashrc for API keys and HF_HOME
import subprocess
result = subprocess.run(['bash', '-c', 'source /home/smallyan/.bashrc && env'], capture_output=True, text=True)
for line in result.stdout.split('\n'):
    if '=' in line:
        key, _, value = line.partition('=')
        if key in ['HF_HOME', 'HF_TOKEN', 'OPENAI_API_KEY', 'NDIF_API_KEY', 'HUGGINGFACE_HUB_CACHE']:
            os.environ[key] = value

print(f"Working directory: {os.getcwd()}")
print(f"HF_HOME: {os.environ.get('HF_HOME', 'NOT SET')}")
print(f"HF_TOKEN available: {'HF_TOKEN' in os.environ}")

Working directory: /home/smallyan/eval_agent
HF_HOME: /net/projects2/chai-lab/shared_models
HF_TOKEN available: True


In [2]:
# Explore the repository structure
repo_path = '/net/scratch2/smallyan/belief-tracking_eval'
for root, dirs, files in os.walk(repo_path):
    # Limit depth
    depth = root.replace(repo_path, '').count(os.sep)
    if depth < 3:
        indent = ' ' * 2 * depth
        print(f'{indent}{os.path.basename(root)}/')
        subindent = ' ' * 2 * (depth + 1)
        for file in files[:20]:  # Limit files shown
            print(f'{subindent}{file}')
        if len(files) > 20:
            print(f'{subindent}... and {len(files) - 20} more files')

In [3]:
# Check if path exists
import os
repo_path = '/net/scratch2/smallyan/belief-tracking_eval'
print(f"Path exists: {os.path.exists(repo_path)}")
if os.path.exists(repo_path):
    print(f"Contents: {os.listdir(repo_path)}")

Path exists: False


In [4]:
# Correct the path - it uses underscores not hyphen
repo_path = '/net/scratch2/smallyan/belief_tracking_eval'
print(f"Path exists: {os.path.exists(repo_path)}")

# Explore the repository structure
for root, dirs, files in os.walk(repo_path):
    depth = root.replace(repo_path, '').count(os.sep)
    if depth < 3:
        indent = ' ' * 2 * depth
        print(f'{indent}{os.path.basename(root)}/')
        subindent = ' ' * 2 * (depth + 1)
        for file in files[:15]:
            print(f'{subindent}{file}')
        if len(files) > 15:
            print(f'{subindent}... and {len(files) - 15} more files')

Path exists: True
belief_tracking_eval/
  plan.md
  env.yml
  pyproject.toml
  uv.lock
  causalmodel_novis.png
  .python-version
  documentation.pdf
  CodeWalkthrough.md
  .gitignore
  data/
    story_templates.json
    synthetic_entities/
      characters.json
      drinks.json
      bottles.json
    bigtom/
  evaluation/
    belief_claude_2026-01-16_02-31-15/
    belief_claude_2026-01-15_22-37-21/
  src/
    dataset.py
    models.txt
    global_utils.py
    __pycache__/
      dataset.cpython-311.pyc
      global_utils.cpython-311.pyc
  no_exe_evaluation/
    code_critic_evaluation.ipynb
    code_critic_summary.json
    generalization_eval.ipynb
    generalization_eval_summary.json
    replications/
      no_exe_evaluation_replication.md
      self_replication_evaluation.json
  scripts/
    evaluate_all_models.py
    evaluate_causalToM.py
    patching_scripts/
      run_patching_exp_utils.py
      run_single_layer_patching_exps.py
      run_upto_layer_patching_exps.py
    tracing_scri

# Generalizability Evaluation for Belief Tracking Repository

## Overview
This notebook evaluates the generalizability of findings in the belief-tracking research repository.

Repository path: `/net/scratch2/smallyan/belief_tracking_eval`

## Evaluation Checklist:
- **GT1**: Generalization to a New Model
- **GT2**: Generalization to New Data  
- **GT3**: Method / Specificity Generalizability

In [5]:
# Read the plan.md to understand the research
with open('/net/scratch2/smallyan/belief_tracking_eval/plan.md', 'r') as f:
    plan_content = f.read()
print(plan_content[:8000])

# Plan
## Objective
Analyze how language models internally represent and track beliefs of characters, particularly when those beliefs may differ from reality, using causal mediation and abstraction methods to uncover systematic computational mechanisms.

## Hypothesis
1. Language models use a lookback mechanism to track beliefs, where reference information is copied to two locations (address and pointer) enabling later retrieval of important information when needed.
2. The model assigns ordering IDs to character, object, and state tokens, then binds character-object-state triples together by co-locating their reference information in low-rank subspaces.
3. A binding lookback retrieves the correct state OI using character and object OIs, and an answer lookback retrieves the corresponding state token.
4. When visibility information is provided, a visibility lookback uses a visibility ID to retrieve information about the observed character and update the observing character's beliefs.

##

In [6]:
# Read the documentation/CodeWalkthrough to understand more details
with open('/net/scratch2/smallyan/belief_tracking_eval/CodeWalkthrough.md', 'r') as f:
    walkthrough = f.read()
print(walkthrough[:10000])

# Language Models use Lookbacks to Track Beliefs

This repository contains the code and experiments for the paper ["Language Models use Lookbacks to Track Beliefs"](https://arxiv.org/abs/2505.14685) by Prakash et al, 2025. The work investigates how language models (specifically Llama-3-70B-Instruct and Llama-3.1-405B-Instruct) represent and track characters' beliefs.


![Causal Model in No-Visibility Setting](causalmodel_novis.png)


Please check [belief.baulab.info](https://belief.baulab.info/) for more information.

## Setup

1. Clone the repository:
```bash
git clone https://github.com/Nix07/belief_tracking.git
cd belief_tracking
```

2. Set up the environment:
```bash
uv sync
source .venv/bin/activate
```

3. Configure `env.yml` with following environment variables:
- Set `NDIF_KEY` for API access
- Set `HF_WRITE` for Hugging Face access

4. To perform subspace level analysis, you would need singular vectors that you can request by sending an email to [Nikhil](https://nix07.github.

In [7]:
# Read the dataset.py to understand data structure
with open('/net/scratch2/smallyan/belief_tracking_eval/src/dataset.py', 'r') as f:
    dataset_code = f.read()
print(dataset_code[:8000])

import json
import os
import random
from dataclasses import dataclass
from typing import Literal

from dataclasses_json import DataClassJsonMixin
from torch.utils.data import Dataset

# from src.utils import env_utils

# Get the absolute path to the data directory
SCRIPT_DIR = os.path.dirname(os.path.abspath(__file__))
PROJECT_ROOT = os.path.dirname(SCRIPT_DIR)
STORY_TEMPLATE_PATH = os.path.join(PROJECT_ROOT, "data", "story_templates.json")

with open(STORY_TEMPLATE_PATH, "r") as f:
    STORY_TEMPLATES = json.load(f)


@dataclass(frozen=False)
class Sample(DataClassJsonMixin):
    template_idx: int
    characters: list[str]
    objects: list[str]
    states: list[str]

    story: str | None = None
    character_belief: list[dict[str, str]] = None

    def __post_init__(self):
        if len(self.characters) == 1:
            self.characters.append("<N/A>")
        # assert len(self.states) == 2 and len(self.containers) == 2 and len(self.characters) == 2
        # No Duplicates
        

In [8]:
# Read the story templates to understand the dataset better
with open('/net/scratch2/smallyan/belief_tracking_eval/data/story_templates.json', 'r') as f:
    templates = json.load(f)
import json
print(json.dumps(templates, indent=2)[:5000])

NameError: name 'json' is not defined

In [9]:
import json
# Read the story templates to understand the dataset better
with open('/net/scratch2/smallyan/belief_tracking_eval/data/story_templates.json', 'r') as f:
    templates = json.load(f)
print(json.dumps(templates, indent=2)[:5000])

{
  "placeholders": {
    "entity": {
      "character": [
        "<character_1>",
        "<character_2>",
        "<character_3>",
        "<character_4>"
      ],
      "container": [
        "<container_1>",
        "<container_2>"
      ],
      "state": [
        "<state_1>",
        "<state_2>"
      ]
    },
    "event": {
      "state_event": "<state_event>",
      "container_event": "<container_event>",
      "state_swap": "<state_swap>"
    },
    "question": {
      "character": "<character_q>",
      "container": "<container_q>",
      "state": "<state_q>"
    },
    "notice": "<noticed/didn't notice>"
  },
  "templates": [
    {
      "context": "<character_1> and <character_2> are working in a busy restaurant. To complete an order, <character_1> grabs an opaque <container_1> and fills it with <state_1>. Then <character_2> grabs another opaque <container_2> and fills it with <state_2>. <character_2> cannot observe <character_1>'s actions. <character_1> cannot observe <ch

In [10]:
# Check which models were used in the original experiments
with open('/net/scratch2/smallyan/belief_tracking_eval/src/models.txt', 'r') as f:
    models_txt = f.read()
print("Models in the research:")
print(models_txt)

# Check results directory for more information about the models used
import os
results_path = '/net/scratch2/smallyan/belief_tracking_eval/results/model_evaluations'
print("\nModel evaluation results:")
print(os.listdir(results_path))

Models in the research:
meta-llama/Llama-2-7b-hf
Qwen/Qwen2.5-7B
Qwen/Qwen2.5-7B-Instruct
meta-llama/Meta-Llama-3-8B
meta-llama/Meta-Llama-3-8B-Instruct
meta-llama/Llama-3.1-8B
meta-llama/Llama-3.1-8B-Instruct
meta-llama/Llama-2-13b-hf
allenai/OLMo-2-1124-13B-Instruct
Qwen/Qwen2.5-14B
Qwen/Qwen2.5-14B-Instruct
google/gemma-3-27b-it
allenai/OLMo-2-0325-32B-Instruct
meta-llama/Meta-Llama-3-70B-Instruct

Model evaluation results:
['Llama-2-7b-hf.json', 'OLMo-2-1124-13B-Instruct_vis.json', 'Qwen2.5-7B-Instruct_vis.json', 'OLMo-2-0325-32B-Instruct.json', 'Qwen2.5-14B-Instruct_vis.json', 'gemma-3-27b-it_vis.json', 'Qwen2.5-14B_vis.json', 'Llama-3.1-8B_vis.json', 'Llama-3.1-8B-Instruct.json', 'Llama-2-7b-hf_vis.json', 'OLMo-2-1124-13B-Instruct.json', 'Meta-Llama-3-8B-Instruct_vis.json', 'Meta-Llama-3-70B-Instruct.json', 'Llama-3.1-8B-Instruct_vis.json', 'Qwen2.5-7B-Instruct.json', 'OLMo-2-0325-32B-Instruct_vis.json', 'Meta-Llama-3-8B-Instruct.json', 'Qwen2.5-7B_vis.json', 'Meta-Llama-3-8B.jso

In [11]:
# Look at sample results from causal mediation analysis to understand the findings
causal_path = '/net/scratch2/smallyan/belief_tracking_eval/results/causal_mediation_analysis'
print("Causal mediation results:")
print(os.listdir(causal_path))

# Read one of the causal mediation results
with open(f'{causal_path}/character.json', 'r') as f:
    character_results = json.load(f)
print("\nCharacter results structure:")
print(json.dumps(character_results, indent=2)[:2000])

Causal mediation results:
['character.json', 'state.json', 'object.json']

Character results structure:
{
  "180": {
    "0": 0.0,
    "1": 0.0,
    "2": 0.0,
    "3": 0.0,
    "4": 0.0,
    "5": 0.0,
    "6": 0.0,
    "7": 0.0,
    "8": 0.0,
    "9": 0.0,
    "10": 0.0,
    "11": 0.0,
    "12": 0.0,
    "13": 0.0,
    "14": 0.0,
    "15": 0.0,
    "16": 0.0,
    "17": 0.0,
    "18": 0.0,
    "19": 0.0,
    "20": 0.0,
    "21": 0.0,
    "22": 0.0,
    "23": 0.0,
    "24": 0.0,
    "25": 0.0,
    "26": 0.0,
    "27": 0.0,
    "28": 0.0,
    "29": 0.0,
    "30": 0.02,
    "31": 0.18,
    "32": 0.9,
    "33": 0.98,
    "34": 1.0,
    "35": 1.0,
    "36": 1.0,
    "37": 1.0,
    "38": 1.0,
    "39": 1.0,
    "40": 1.0,
    "41": 1.0,
    "42": 1.0,
    "43": 1.0,
    "44": 1.0,
    "45": 1.0,
    "46": 1.0,
    "47": 1.0,
    "48": 1.0,
    "49": 1.0,
    "50": 1.0,
    "51": 1.0,
    "52": 1.0,
    "53": 1.0,
    "54": 1.0,
    "55": 1.0,
    "56": 1.0,
    "57": 1.0,
    "58": 1.0,
    "

In [12]:
# Check the notebooks for the core analysis - look at the causalToM_novis experiments
notebook_path = '/net/scratch2/smallyan/belief_tracking_eval/notebooks/causalToM_novis'
print("causalToM_novis notebooks:")
print(os.listdir(notebook_path))

# Also check the tracing scripts to understand the methodology
scripts_path = '/net/scratch2/smallyan/belief_tracking_eval/scripts'
print("\nScripts:")
for root, dirs, files in os.walk(scripts_path):
    for f in files:
        print(f"  {os.path.join(root, f)}")

causalToM_novis notebooks:
['__pycache__', 'binding_lookback.ipynb', 'answer_lookback.ipynb', 'utils.py']

Scripts:
  /net/scratch2/smallyan/belief_tracking_eval/scripts/evaluate_all_models.py
  /net/scratch2/smallyan/belief_tracking_eval/scripts/evaluate_causalToM.py
  /net/scratch2/smallyan/belief_tracking_eval/scripts/patching_scripts/run_patching_exp_utils.py
  /net/scratch2/smallyan/belief_tracking_eval/scripts/patching_scripts/run_single_layer_patching_exps.py
  /net/scratch2/smallyan/belief_tracking_eval/scripts/patching_scripts/run_upto_layer_patching_exps.py
  /net/scratch2/smallyan/belief_tracking_eval/scripts/patching_scripts/__pycache__/run_patching_exp_utils.cpython-311.pyc
  /net/scratch2/smallyan/belief_tracking_eval/scripts/tracing_scripts/utils.py
  /net/scratch2/smallyan/belief_tracking_eval/scripts/tracing_scripts/trace.py
  /net/scratch2/smallyan/belief_tracking_eval/scripts/tracing_scripts/__pycache__/utils.cpython-311.pyc


In [13]:
# Read the tracing scripts to understand the causal mediation methodology
with open('/net/scratch2/smallyan/belief_tracking_eval/scripts/tracing_scripts/trace.py', 'r') as f:
    trace_code = f.read()
print(trace_code[:6000])

import json
import os
import random
import sys
from dataclasses import dataclass
from typing import Callable

import fire
from nnsight import CONFIG
from torch.utils.data import DataLoader
from utils import (
    find_correct_samples,
    get_character_tracing_exps,
    get_object_tracing_exps,
    get_state_tracing_exps,
    load_entity_data,
    load_model,
    run_tracing_experiment,
)

# Add project root to path before importing from src
project_root = os.path.dirname(
    os.path.dirname(os.path.dirname(os.path.abspath(__file__)))
)
sys.path.append(project_root)
from src import global_utils

# Get credentials from environment variables
nnsight_api_key = global_utils.load_env_var("NDIF_KEY")
hf_token = global_utils.load_env_var("HF_WRITE")

# Set credentials
CONFIG.set_default_api_key(nnsight_api_key)
os.environ["HF_TOKEN"] = hf_token
logger = global_utils.logger


@dataclass
class Tracer:
    """Class to run Causal Mediation Analysis experiments for tracing the information flow.""

In [14]:
# Read the utils for tracing to understand the intervention methodology
with open('/net/scratch2/smallyan/belief_tracking_eval/scripts/tracing_scripts/utils.py', 'r') as f:
    utils_code = f.read()
print(utils_code[:8000])

"""
Utility functions for causal mediation analysis experiments
"""

import json
import os
import random
import sys
from collections import defaultdict

import torch
from nnsight import LanguageModel
from torch.utils.data import DataLoader
from tqdm import tqdm

# Add project root to path before importing from src
project_root = os.path.dirname(
    os.path.dirname(os.path.dirname(os.path.abspath(__file__)))
)
sys.path.append(project_root)

from src.dataset import Dataset, Sample


def load_model(
    model_name: str, is_remote: bool = False, cache_dir: str = None
) -> LanguageModel:
    """Load the language model and tokenizer as LanguageModel object.

    Args:
        model_name: Name of the model to use
        is_remote: Whether to run model inference remotely
        cache_dir: Directory to cache model files

    Returns:
        LanguageModel object
    """

    if is_remote:
        model = LanguageModel(model_name)
    else:
        model = LanguageModel(
            model_nam

## Understanding the Original Research

### Key Findings:
1. **Lookback Mechanism**: Language models use a lookback mechanism to track beliefs, where reference information is copied to two locations (address and pointer).

2. **Binding Mechanism**: The model assigns ordering IDs to character, object, and state tokens, then binds character-object-state triples together in low-rank subspaces.

3. **Key Layer Findings** (from Llama-3-70B-Instruct):
   - Answer payload localizes at layers 56+ at final token
   - Answer pointer at layers 34-52 at final token
   - Binding address and payload at layers 33-38 at state token
   - Binding source reference (character/object) at layers 20-34
   - Visibility ID source at layers 10-23

### Models Used in Original Work:
- Llama-3-70B-Instruct (primary)
- Llama-3.1-405B-Instruct (validation)
- Plus various smaller models for model evaluation

### Methodology:
- **Causal Mediation Analysis**: Interchange interventions to trace information flow
- **Interchange Intervention Accuracy (IIA)**: Metric for measuring whether output changes as expected when intervening

In [15]:
# Check available models in the HF cache
import os
hf_cache = '/net/projects2/chai-lab/shared_models/hub'
print("Available cached models:")
models = os.listdir(hf_cache)
for m in sorted(models):
    if 'llama' in m.lower() or 'mistral' in m.lower() or 'qwen' in m.lower() or 'gemma' in m.lower():
        print(f"  {m}")

Available cached models:
  383751c9e07fcd41ec1b3b32a88d2c08952b9f241eb94977b5ef66b566ba3d4cmeta-llama-Llama-3.3-70B-Instruct.lock
  models--Qwen--QwQ-32B
  models--Qwen--Qwen2-0.5B
  models--Qwen--Qwen2.5-0.5B
  models--Qwen--Qwen2.5-0.5B-Instruct
  models--Qwen--Qwen2.5-1.5B
  models--Qwen--Qwen2.5-32B-Instruct
  models--Qwen--Qwen2.5-7B-Instruct
  models--Qwen--Qwen3-0.6B
  models--Qwen--Qwen3-32B
  models--deepseek-ai--DeepSeek-R1-Distill-Qwen-32B
  models--google--gemma-2-27b-it
  models--google--gemma-2-2b
  models--google--gemma-2-2b-it
  models--google--gemma-2-9b-it
  models--google--gemma-2b
  models--meta-llama--Llama-2-7b-hf
  models--meta-llama--Llama-3.1-8B-Instruct
  models--meta-llama--Llama-3.2-3B
  models--meta-llama--Llama-3.3-70B-Instruct
  models--meta-llama--Meta-Llama-3-70B-Instruct
  models--meta-llama--Meta-Llama-3-8B
  models--meta-llama--Meta-Llama-3-8B-Instruct
  models--meta-llama--Meta-Llama-3.1-8B-Instruct
  models--mistralai--Mistral-7B-Instruct-v0.2
  mo

In [16]:
# Check CUDA availability
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Number of GPUs: {torch.cuda.device_count()}")
    for i in range(torch.cuda.device_count()):
        print(f"  GPU {i}: {torch.cuda.get_device_name(i)}")
        print(f"    Memory: {torch.cuda.get_device_properties(i).total_memory / 1e9:.1f} GB")

CUDA available: True
Number of GPUs: 1
  GPU 0: NVIDIA A40
    Memory: 47.7 GB


---

## GT1: Generalization to a New Model

The original work used Llama-3-70B-Instruct and Llama-3.1-405B-Instruct as primary models. We will test whether the lookback mechanism findings generalize to:

**New Model: Mistral-7B-Instruct-v0.3**

This model:
- Was NOT used in the original work
- Is from a different model family (Mistral vs Llama)
- Has a different architecture

We will run causal mediation analysis to test if similar layer patterns emerge for:
1. Character information localization
2. Object information localization
3. State information localization

In [17]:
# Set up the environment and load the dataset module
import sys
sys.path.insert(0, '/net/scratch2/smallyan/belief_tracking_eval')
sys.path.insert(0, '/net/scratch2/smallyan/belief_tracking_eval/src')

from src.dataset import Dataset, Sample
import json
import random

# Load entity data
data_dir = '/net/scratch2/smallyan/belief_tracking_eval/data'

with open(f'{data_dir}/synthetic_entities/characters.json', 'r') as f:
    all_characters = list(json.load(f))
with open(f'{data_dir}/synthetic_entities/bottles.json', 'r') as f:
    all_objects = list(json.load(f))
with open(f'{data_dir}/synthetic_entities/drinks.json', 'r') as f:
    all_states = list(json.load(f))

print(f"Loaded {len(all_characters)} characters, {len(all_objects)} objects, {len(all_states)} states")
print(f"Sample characters: {all_characters[:3]}")
print(f"Sample objects: {all_objects[:3]}")
print(f"Sample states: {all_states[:3]}")

Loaded 103 characters, 21 objects, 23 states
Sample characters: ['Dean', 'Beth', 'Jake']
Sample objects: ['jar', 'cup', 'mug']
Sample states: ['water', 'milk', 'tea']


In [18]:
# Create a sample dataset item
random.seed(42)
sample_characters = random.sample(all_characters, 2)
sample_objects = random.sample(all_objects, 2)
sample_states = random.sample(all_states, 2)

sample = Sample(
    template_idx=0,  # Template 0: Both characters can't observe each other
    characters=sample_characters,
    objects=sample_objects,
    states=sample_states,
)

dataset = Dataset(samples=[sample])
item = dataset.__getitem__(0, set_character=0, set_container=0)

print("Sample prompt:")
print(item['prompt'])
print(f"\nExpected answer: {item['target']}")

Sample prompt:
Instruction: 1. Track the belief of each character as described in the story. 2. A character's belief is formed only when they perform an action themselves or can observe the action taking place. 3. A character does not have any beliefs about the container and its contents which they cannot observe. 4. To answer the question, predict only what is inside the queried container, strictly based on the belief of the character, mentioned in the question. 5. If the queried character has no belief about the container in question, then predict 'unknown'. 6. Do not predict container or character as the final output.

Story: Charlie and Pete are working in a busy restaurant. To complete an order, Charlie grabs an opaque jar and fills it with wine. Then Pete grabs another opaque can and fills it with soda. Pete cannot observe Charlie's actions. Charlie cannot observe Pete's actions.
Question: What does Charlie believe the jar contains?
Answer:

Expected answer: wine


In [19]:
# Load Mistral-7B-Instruct-v0.3 for GT1 testing
# This model was NOT used in the original research
from nnsight import LanguageModel
import torch

hf_cache = '/net/projects2/chai-lab/shared_models/hub'

print("Loading Mistral-7B-Instruct-v0.3...")
model = LanguageModel(
    'mistralai/Mistral-7B-Instruct-v0.3',
    device_map='auto',
    torch_dtype=torch.float16,
    dispatch=True,
    cache_dir=hf_cache,
)
print("Model loaded successfully!")
print(f"Number of layers: {len(model.model.layers)}")

Loading Mistral-7B-Instruct-v0.3...


`torch_dtype` is deprecated! Use `dtype` instead!


tokenizer.model:   0%|          | 0.00/587k [00:00<?, ?B/s]

In [20]:
# Test if the model can do belief tracking task first
prompt = item['prompt']
with model.trace(prompt) as tracer:
    output = model.lm_head.output.save()
    
# Get logits and prediction
logits = output.value[0, -1, :]
top_token = torch.argmax(logits).item()
predicted_token = model.tokenizer.decode([top_token])
print(f"Predicted next token: '{predicted_token}'")
print(f"Expected: '{item['target']}'")
print(f"Match: {item['target'].strip() in predicted_token.strip()}")

In [21]:
# Check model structure
print(f"Model type: {type(model.model)}")
print(f"Number of layers: {len(model.model.layers)}")
print(f"Hidden size: {model.model.config.hidden_size}")

In [22]:
# Let me check if the output was saved
print(type(output))
print(output)

In [23]:
# Something seems to be wrong with output display - let's try direct generation
tokens = model.tokenizer(prompt, return_tensors='pt').to(model.device)
with torch.no_grad():
    outputs = model.model.generate(
        tokens['input_ids'],
        max_new_tokens=5,
        do_sample=False,
        pad_token_id=model.tokenizer.pad_token_id if model.tokenizer.pad_token_id else model.tokenizer.eos_token_id
    )
    
generated = model.tokenizer.decode(outputs[0][len(tokens['input_ids'][0]):], skip_special_tokens=True)
print(f"Generated: '{generated}'")
print(f"Expected: '{item['target']}'")
print(f"Correct: {item['target'].strip().lower() in generated.strip().lower()}")

In [24]:
# Debug - check basic output 
print("Testing basic output...")
x = 1 + 1
print(f"x = {x}")